In [1]:
import pandas as pd
import time
import requests
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings('ignore')

# Scrape S&P 500 tickers from Wikipedia
def get_sp500_tickers():
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    table = soup.find('table', {'id': 'constituents'})
    tickers = []
    for row in table.find_all('tr')[1:]:
        ticker = row.find_all('td')[0].text.strip().replace('.', '-')  # Normalize tickers (e.g., BRK.B → BRK-B)
        tickers.append(ticker)
    return sorted(tickers)

def map_tickers_to_figis(tickers, api_key):
    headers = {
        'Content-Type': 'application/json',
        'X-OPENFIGI-APIKEY': api_key
    }
    url = 'https://api.openfigi.com/v3/mapping'
    results = []

    # OpenFIGI allows up to 100 tickers per request
    for i in range(0, len(tickers), 100):
        batch = tickers[i:i+100]
        mapping_jobs = [{'idType': 'TICKER', 'idValue': ticker, 'exchCode': 'US'} for ticker in batch]
        response = requests.post(url, headers=headers, json=mapping_jobs)
        if response.status_code == 200:
            data = response.json()
            for item in data:
                if 'data' in item and item['data']:
                    for entry in item['data']:
                        results.append({
                            'ticker': entry.get('ticker'),
                            'cusip': entry.get('cusip'),
                            'name': entry.get('name'),
                            'figi': entry.get('figi')
                        })
        else:
            print(f"Error: {response.status_code} - {response.text}")
        time.sleep(1)  # To respect API rate limits

    override = {
        'BRK-B': 'BBG000DWG505',   # FIGI for Berkshire Hathaway Class B :contentReference[oaicite:0]{index=0}
        'BF-B':  'BBG001S5P4H1'    # FIGI for Brown-Forman Class B :contentReference[oaicite:1]{index=1}
        }
    # After you collect `results` from OpenFIGI:
    for tick in ['BRK-B', 'BF-B']:
        results.append({
            'ticker': tick,
            'cusip': None,    # if you don’t have CUSIP, leave it None or fill in
            'name': None,     # same for name
            'figi': override[tick]
        })
    return results

# Initialize counters and mappings
total_edges = 0
filtered_edges = 0
unique_ciks = set()
unique_cusips = set()

# Get S&P 500 tickers
sp500_tickers = get_sp500_tickers()
print(f"Fetched {len(sp500_tickers)} S&P 500 tickers")

api_key = '8c4a6a49-dc82-4baf-bdb2-27e32ab5c717'
sp500_map = map_tickers_to_figis(sp500_tickers, api_key)
sp500_figis = sorted([item['figi'] for item in sp500_map])

# Filter dataset for S&P 500 FIGIs
chunk_size = 10000000
filtered_chunks = []
for chunk in pd.read_csv('merged_dataset_upper.csv', chunksize=chunk_size):
    unique_ciks.update(chunk['CIK'])
    unique_cusips.update(chunk['CUSIP'])
    chunk_edges = len(chunk)
    total_edges += chunk_edges
    # Filter for S&P 500 FIGIs
    filtered_chunk = chunk[chunk['FIGI'].isin(sp500_figis)]
    filtered_edges += len(filtered_chunk)
    if not filtered_chunk.empty:
        filtered_chunks.append(filtered_chunk)
    print(f"Processed chunk with {chunk_edges} rows, kept {len(filtered_chunk)} S&P 500 rows")

# Save filtered dataset
if filtered_chunks:
    filtered_df = pd.concat(filtered_chunks)
    filtered_df.to_csv('sp500_figi.csv', index=False)
    print(f"Saved {filtered_edges} rows to sp500_figi.csv")


# Print results
print(f"Total edges: {total_edges}")
print(f"Filtered S&P 500 edges: {filtered_edges}")
print(f"Unique CIKs: {len(unique_ciks)}")
print(f"S&P 500 CIKs: {len(filtered_df['CIK'].unique())}")
print(f"Unique CUSIPs: {len(unique_cusips)}")
sp500_cusips = filtered_df['CUSIP'].unique()
print(f"S&P 500 CUSIPs: {len(sp500_cusips)}")

# Inspect sample chunk
sample_chunk = pd.read_csv('sp500_figi.csv', nrows=1000)
print("\nSample chunk head:")
print(sample_chunk.head())

print(f"\nUnique FIGIs: {len(filtered_df['FIGI'].unique())}")
print(f"Unique CUSIPs: {len(sp500_cusips)}\n")

total_edges = 0
filtered_edges = 0
unique_ciks = set()
unique_cusips = set()

filtered_chunks = []
for chunk in pd.read_csv('merged_dataset_upper.csv', chunksize=chunk_size):
    unique_ciks.update(chunk['CIK'])
    unique_cusips.update(chunk['CUSIP'])
    chunk_edges = len(chunk)
    total_edges += chunk_edges
    # Filter for S&P 500 CUSIPs
    filtered_chunk = chunk[chunk['CUSIP'].isin(sp500_cusips)]
    filtered_edges += len(filtered_chunk)
    if not filtered_chunk.empty:
        filtered_chunks.append(filtered_chunk)
    print(f"Processed chunk with {chunk_edges} rows, kept {len(filtered_chunk)} S&P 500 rows")

# Save filtered dataset
if filtered_chunks:
    filtered_df = pd.concat(filtered_chunks)
    filtered_df.to_csv('sp500.csv', index=False)
    print(f"Saved {filtered_edges} rows to sp500.csv")

# Print results
print(f"Total edges: {total_edges}")
print(f"Filtered S&P 500 edges: {filtered_edges}")
print(f"Unique CIKs: {len(unique_ciks)}")
print(f"S&P 500 CIKs: {len(filtered_df['CIK'].unique())}")
print(f"Unique CUSIPs: {len(unique_cusips)}")
print(f"S&P 500 CUSIPs: {len(filtered_df['CUSIP'].unique())}")
print(f"Unique FIGIs: {len(filtered_df['FIGI'].unique())}")

Fetched 503 S&P 500 tickers
Processed chunk with 10000000 rows, kept 0 S&P 500 rows
Processed chunk with 10000000 rows, kept 0 S&P 500 rows
Processed chunk with 10000000 rows, kept 0 S&P 500 rows
Processed chunk with 10000000 rows, kept 0 S&P 500 rows
Processed chunk with 10000000 rows, kept 0 S&P 500 rows
Processed chunk with 10000000 rows, kept 0 S&P 500 rows
Processed chunk with 10000000 rows, kept 41090 S&P 500 rows
Processed chunk with 10000000 rows, kept 54210 S&P 500 rows
Processed chunk with 3859935 rows, kept 34085 S&P 500 rows
Saved 129385 rows to sp500_figi.csv
Total edges: 83859935
Filtered S&P 500 edges: 129385
Unique CIKs: 11504
S&P 500 CIKs: 230
Unique CUSIPs: 87427
S&P 500 CUSIPs: 592

Sample chunk head:
       ACCESSION_NUMBER FILING_DATE SUBMISSIONTYPE    CIK PERIODOFREPORT  \
0  0000053417-23-000020  2023-02-09         13F-HR  53417     2022-12-31   
1  0000053417-23-000020  2023-02-09         13F-HR  53417     2022-12-31   
2  0000053417-23-000020  2023-02-09       